## Conceptual Overview of LFM2Audio

### Architecture
- **LFM2 Backbone**: Autoregressive transformer (Lfm2Model) that generates text and audio tokens sequentially.
- **Conformer Encoder**: Processes input audio (log-mel features) into embeddings that feed into the LFM2 backbone.
- **Audio Adapter**: Projects Conformer outputs to LFM2 hidden dimension.
- **Depthformer**: Generates audio codebooks token-by-token using a shallow transformer; processes codebooks 0-7 in sequence.
- **Shared Embeddings**: Text and audio embeddings share vocabulary space with tieing options.

### Modalities
- **TEXT (0)**: Input/Output text tokens from the tokenizer.
- **AUDIO_IN (1)**: Input audio embeddings (user audio converted to embeddings).
- **AUDIO_OUT (2)**: Output audio tokens (model-generated audio codes).

### Generation Modes
- **Sequential (`generate_sequential`)**: Generates all text until `<|audio_start|>`, then all audio tokens until `<|EOAudio|>`, repeating.
- **Interleaved (`generate_interleaved`)**: Generates `interleaved_n_text` text tokens, then `interleaved_n_audio` audio tokens, switching back and forth based on control tokens (`<|text_end|>` and `<|EOAudio|>`).

### Audio Tokenization
- **8 Codebooks**: Audio is encoded as 8 parallel codebooks (like EnCodec/Mimi).
- **Vocabulary**: 2048 tokens per codebook + 1 end-of-audio token (2048).
- **Semantic Weighting**: First codebook (semantic) gets higher loss weight (`semantic_codebook_factor`).
- **Depthformer**: Generates all 8 codebook tokens for each audio frame in order.

### Processing Pipeline
- **Input**: `ChatState` with text/audio inputs is converted to tensors (`text`, `audio_in`, `audio_out`, `modality_flag`).
- **Prefill**: Embeddings are assembled in order specified by `modality_flag` and fed to LFM2.
- **Generation**: Autoregressive decoding with KV-cache, yielding one token (text) or 8 tokens (audio frame) per step.
- **Detokenization**: Audio token matrix `(B, frames, 8)` is decoded to waveform via the processor's `decode()` method.

### Key Configuration
- `codebooks`: 8 (fixed by Mimi/EnCodec).
- `interleaved_n_text/audio`: Control token counts per block in interleaved mode.
- `semantic_codebook_factor`: Weight for first codebook in loss (typically > 1 for more semantic focus).


Simplest example

In [ ]:
# ASR
import torch
import torchaudio
from src.liquid_audio import LFM2AudioModel, LFM2AudioProcessor, ChatState

# Load models
HF_REPO = "LiquidAI/LFM2.5-Audio-1.5B"

processor = LFM2AudioProcessor.from_pretrained(HF_REPO).eval()
model = LFM2AudioModel.from_pretrained(HF_REPO).eval()

# Set up inputs for the model
chat = ChatState(processor)

chat.new_turn("system")
chat.add_text("Perform ASR.")
chat.end_turn()

chat.new_turn("user")
wav, sampling_rate = torchaudio.load("assets/asr.wav")
chat.add_audio(wav, sampling_rate)
chat.end_turn()

chat.new_turn("assistant")

# Generate text
for t in model.generate_sequential(**chat, max_new_tokens=512):
    if t.numel() == 1: #? t is just a long tensor like 
        print(processor.text.decode(t), end="", flush=True)


# TTS

chat.new_turn("system")
chat.add_text("Perform TTS. Use the UK male voice.")
chat.end_turn()

chat.new_turn("user")
chat.add_text("What is this obsession people have with books? They put them in their houses—like they're trophies. What do you need it for after you read it?")
chat.end_turn()

chat.new_turn("assistant")

# Generate text
audio_out: list[torch.Tensor] = []
for t in model.generate_sequential(**chat, max_new_tokens=512, audio_temperature = 0.8, audio_top_k=64):
    if t.numel() > 1:
        audio_out.append(t)

# Detokenize audio
audio_codes = torch.stack(audio_out[:-1], 1).unsqueeze(0) #? audio_out[:-1] drops the last output element t, which is for end of audio. Then we stack all the t tensors along the second dim, and then put an out dimension as well.
waveform = processor.decode(audio_codes)
torchaudio.save("tts.wav", waveform.cpu(), 24_000)


In [ ]:
from liquid_audio import ChatState, LFM2AudioProcessor, LFM2AudioModel

config = LFM2AudioProcessor("ckpt")

model = LFM2AudioModel(config)

chat = ChatState(config)

chat.new_turn(r"system | user | assistant")

chat.add_text("message...")
# or
wav, sampling_rate = torchaudio.load("assets/asr.wav")
chat.add_audio(wav, sampling_rate)

chat.end_turn()

for t in model.generate_sequential(**chat, max_new_tokens=512):
    if t.numel() == 1: #? t.numel() >= 1 for audio
        print(processor.text.decode(t), end="", flush=True)


1. Entry point of the model is model.generate_sequential(**chat, max_new_tokens=512, audio_temperature = 0.8, audio_top_k=64)
2. Input depends on `chat.add_text("text)"` or `chat.add_audio(wav, sr)`
3. The `generate_sequential` method returns a generator which is looped through and when a for loop is used, a tensor is yielded at every iteration
    - If the tensor has only one element -> it is maybe a long tensor (like [256]) and then it is decoded to the word chunk.
    - If t.numel() > 1 -> then all these t tensors are saved in a list and then all of them except the last token (maybe end of audio token) is stacked along the 1st dimension. Finally this tensor is passed into processor.decode to get the complete waveform
    ```python
    audio_codes = torch.stack(audio_out[:-1], 1).unsqueeze(0)
    waveform = processor.decode(audio_codes)
    torchaudio.save("tts.wav", waveform.cpu(), 24_000)
    ```

In [ ]:
import torch
import torchaudio
from liquid_audio import LFM2AudioModel, LFM2AudioProcessor, ChatState, LFMModality

# Load models
HF_REPO = "LiquidAI/LFM2.5-Audio-1.5B"

processor = LFM2AudioProcessor.from_pretrained(HF_REPO).eval()
model = LFM2AudioModel.from_pretrained(HF_REPO).eval()

# Set up inputs for the model
chat = ChatState(processor)

chat.new_turn("system")
chat.add_text("Respond with interleaved text and audio.") #! it's important to instruct this explicitely
chat.end_turn()

chat.new_turn("user")
wav, sampling_rate = torchaudio.load("assets/question.wav")
chat.add_audio(wav, sampling_rate)
chat.end_turn()

chat.new_turn("assistant")

# Generate text and audio tokens.
text_out: list[torch.Tensor] = []
audio_out: list[torch.Tensor] = []
modality_out: list[LFMModality] = []
for t in model.generate_interleaved(**chat, max_new_tokens=512, audio_temperature=1.0, audio_top_k=4):
    if t.numel() == 1:
        print(processor.text.decode(t), end="", flush=True)
        text_out.append(t)
        modality_out.append(LFMModality.TEXT)
    else:
        audio_out.append(t)
        modality_out.append(LFMModality.AUDIO_OUT)

# output: Sure! How about "Handcrafted Woodworking, Precision Made for You"? Another option could be "Quality Woodworking, Quality Results." If you want something more personal, you might try "Your Woodworking Needs, Our Expertise."

# Detokenize audio, removing the last "end-of-audio" codes
# Mimi returns audio at 24kHz
audio_codes = torch.stack(audio_out[:-1], 1).unsqueeze(0)
waveform = processor.decode(audio_codes)
torchaudio.save("answer1.wav", waveform.cpu(), 24_000)

# Append newly generated tokens to chat history
chat.append(
    text = torch.stack(text_out, 1),
    audio_out = torch.stack(audio_out, 1),
    modality_flag = torch.tensor(modality_out),
)
chat.end_turn()

# Start new turn
chat.new_turn("user")
chat.add_text("My business specialized in chairs, can you give me something related to that?")
chat.end_turn()

chat.new_turn("assistant")

# Generate second turn text and audio tokens.
audio_out: list[torch.Tensor] = []
for t in model.generate_interleaved(**chat, max_new_tokens=512, audio_temperature=1.0, audio_top_k=4):
    if t.numel() == 1:
        print(processor.text.decode(t), end="", flush=True)
    else:
        audio_out.append(t)

# output: Sure thing! How about “Comfortable Chairs, Crafted with Care” or “Elegant Seats, Handcrafted for You”? Let me know if you’d like a few more options.

# Detokenize second turn audio, removing the last "end-of-audio" codes
audio_codes = torch.stack(audio_out[:-1], 1).unsqueeze(0)
waveform = processor.decode(audio_codes)
torchaudio.save("answer2.wav", waveform.cpu(), 24_000)

Hands on demo

output transcript for asr.wav:
The stale smell of old beer lingers. It takes heat to bring out the odor. A cold dip restores health and zest. A salt pickle tastes fine with ham. Tacos al pastor are my favorite. A zestful food is the hot cross bun.<|im_end|>% 

2/11 7pm
Starting point
1. model = LFM2AudioModel.from_pretrained(ckpt)
2. model.generate_sequential(**chat, max_new_tokens=512) for audio, we can also pass audio_temperature=1.0, audio_top_k=4

LFM2AudioModel
    __init__
    from_pretrained (cls method)
    generate_sequential
        in_emb = self._prefill()
        lfm_out = self.lfm(in_emb) # lfm is lfm2 model 
        
    generate_interleaved
    _prefill
    _sample_text_token
    _sample_audio_frame

check the ChatState class